# 05 — Exercise solutions: text classification

The cells below give runnable answers. Read each derivation and compare its assumptions with your own.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

## 1. Smoothed IDF and normalized rows

For $n=10$ and $df=3$, the value is $\log((1+10)/(1+3))+1=\log(11/4)+1approximately 2.011$. A ubiquitous term has $df=10$, so its value is $\log(11/11)+1=1$. If $df=0$, smoothing gives $\log(11/1)+1$, which is finite. The vector $(3,4)$ has norm $5$ and unit vector $(0.6,0.8)$.

In [ ]:
idf = np.log(11 / 4) + 1
ubiquitous_idf = np.log(11 / 11) + 1
absent_term_idf = np.log(11 / 1) + 1
unit = np.array([3.0, 4.0]) / 5
print(idf, ubiquitous_idf, absent_term_idf, unit, np.linalg.norm(unit))

## 2. Cosine similarity

If $\|u\|_2=\|v\|_2=1$, the denominator in $(u^Tv)/(\|u\|_2\|v\|_2)$ is one, leaving $u^Tv$. For $(1,0,1)$ and $(1,1,0)$, the dot product is 1 and both norms are $\sqrt2$, so cosine is $1/2$. Sparse storage avoids work on coordinates that are zero in both documents.

In [ ]:
u = np.array([1.0, 0.0, 1.0])
v = np.array([1.0, 1.0, 0.0])
cosine = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
assert np.isclose(cosine, 0.5)
print(cosine)

## 3. Softmax cross-entropy derivative

Softmax gives $\partial p_r/\partial z_k=p_r(\mathbf{1}_{r=k}-p_k)$. Then

$$\frac{\partial}{\partial z_k}\left(-\sum_r y_r\log p_r\right)
=-\sum_r y_r(\mathbf{1}_{r=k}-p_k)=p_k-y_k,$$

because a one-hot target sums to one. Since $z_{ik}=w_k^Tx_i+b_k$, the batch slope gradient is $n^{-1}\sum_i(p_{ik}-y_{ik})x_i$. An L2 penalty discourages large slopes and reduces variance; scikit-learn's inverse strength `C` controls how strongly it does so.

In [ ]:
logits = np.array([0.2, -0.1, 0.7])
probabilities = np.exp(logits - logits.max())
probabilities /= probabilities.sum()
y = np.array([0.0, 1.0, 0.0])
print(probabilities - y)

## 4. Group-aware split

Shuffle the 12 group IDs independently within each category with the fixed seed, then allocate 8, 2, and 2 groups. Since every group contributes eight rows, each category contributes 64 training rows, 16 validation rows, and 16 test rows. A row split can put close template siblings on both sides and reward pattern memorization.

In [ ]:
for candidate in (Path.cwd(), *Path.cwd().parents):
    data_path = candidate / "data" / "support_tickets.csv"
    if data_path.exists():
        break
tickets = pd.read_csv(data_path)
rng = random.Random(2026)
split_groups = {"train": [], "validation": [], "test": []}
for category in sorted(tickets.category.unique()):
    groups = sorted(
        tickets.loc[tickets.category == category, "template_group"].unique()
    )
    rng.shuffle(groups)
    split_groups["train"].extend(groups[:8])
    split_groups["validation"].extend(groups[8:10])
    split_groups["test"].extend(groups[10:12])
splits = {
    name: tickets[tickets.template_group.isin(groups)].reset_index(drop=True)
    for name, groups in split_groups.items()
}
assert set(split_groups["train"]).isdisjoint(split_groups["validation"])
assert set(split_groups["train"]).isdisjoint(split_groups["test"])
assert set(split_groups["validation"]).isdisjoint(split_groups["test"])
for name, frame in splits.items():
    counts = frame.category.value_counts().sort_index()
    assert counts.nunique() == 1
    print(name, len(frame), counts.to_dict())

## 5. Validation selection and one final test

This answer follows the same frozen selection procedure as the lesson. The validation scores may vary with package versions, but the selection rule and group assignment are deterministic. The test set is evaluated only after choosing `C`; do not change that choice based on the displayed test results.

In [ ]:
candidate_C = [0.05, 0.2, 1.0, 5.0, 20.0]
rows = []
for C in candidate_C:
    model = Pipeline(
        [
            ("tfidf", TfidfVectorizer()),
            ("logistic", LogisticRegression(C=C, solver="lbfgs", max_iter=1500)),
        ]
    )
    model.fit(splits["train"].text, splits["train"].category)
    pred = model.predict(splits["validation"].text)
    rows.append(
        {
            "C": C,
            "macro_f1": f1_score(splits["validation"].category, pred, average="macro"),
        }
    )
scores = pd.DataFrame(rows)
display(scores)
best_C = float(scores.sort_values(["macro_f1", "C"], ascending=[False, True]).iloc[0].C)
train_validation = pd.concat([splits["train"], splits["validation"]], ignore_index=True)
final_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("logistic", LogisticRegression(C=best_C, solver="lbfgs", max_iter=1500)),
    ]
)
final_model.fit(train_validation.text, train_validation.category)
test_prediction = final_model.predict(splits["test"].text)
print("chosen C:", best_C)
print("test accuracy:", accuracy_score(splits["test"].category, test_prediction))
print(
    "test macro-F1:",
    f1_score(splits["test"].category, test_prediction, average="macro"),
)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(train_validation.text, train_validation.category)
baseline_prediction = baseline.predict(splits["test"].text)
print(
    "most-frequent baseline accuracy:",
    accuracy_score(splits["test"].category, baseline_prediction),
)
print(
    "most-frequent baseline macro-F1:",
    f1_score(splits["test"].category, baseline_prediction, average="macro"),
)
wrong = splits["test"].assign(predicted=test_prediction).query("category != predicted")
display(wrong[["template_group", "category", "predicted", "text"]].head(2))

## 6. Coefficients and limits

Softmax is invariant to adding the same vector to every class. The interpretable pairwise log-odds use the contrast `(w_k - w_l)^T x + (b_k - b_l)`. That contrast depends on vocabulary, other terms, regularization, and training data; it does not show that a word causes a queue assignment or customer outcome. The synthetic test cannot assess natural language variation, time drift, class imbalance, language differences, sensitive attributes, or queue-specific error costs.

In [ ]:
feature_names = final_model.named_steps["tfidf"].get_feature_names_out()
logistic = final_model.named_steps["logistic"]
class_index = list(logistic.classes_).index("billing")
weights = logistic.coef_[class_index]
top = np.argsort(weights)[-5:][::-1]
print(list(zip(feature_names[top], weights[top].round(3))))